# Microstructure Factor Zoo — AAPL
**MFE 412 — Final Project**

Compute 12 L2 microstructure features per 1-minute bar and use OLS + gradient boosting
to predict the 1-minute forward midprice return. Ablation study shows which features carry signal.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('data/raw')
TRAIN_DATES = [
    '2024-01-02','2024-01-04','2024-01-08','2024-01-10','2024-01-12',
    '2024-01-16','2024-01-18','2024-01-22','2024-01-24','2024-01-26',
    '2024-02-01','2024-02-05','2024-02-07','2024-02-09','2024-02-12',
    '2024-02-14','2024-02-20','2024-02-22','2024-02-26','2024-02-28',
]
TEST_DATES = [
    '2024-03-04','2024-03-07','2024-03-11','2024-03-14','2024-03-18',
    '2024-04-01','2024-04-04','2024-04-08','2024-04-11','2024-04-15',
]

## 1. Load Data

In [ ]:
def load_mbp10(dates: list) -> pd.DataFrame:
    frames = []
    for d in dates:
        p = DATA_DIR / 'AAPL' / 'mbp-10' / f'{d}.parquet'
        if not p.exists():
            print(f'Missing: {p}')
            continue
        frames.append(pd.read_parquet(p).sort_index())
    if not frames:
        raise FileNotFoundError('No AAPL MBP-10 data. Run download_data.py first.')
    return pd.concat(frames)


def load_trades(dates: list) -> pd.DataFrame:
    frames = []
    for d in dates:
        p = DATA_DIR / 'AAPL' / 'trades' / f'{d}.parquet'
        if not p.exists():
            continue
        frames.append(pd.read_parquet(p).sort_index())
    return pd.concat(frames) if frames else pd.DataFrame()


print('Loading MBP-10...')
book_train  = load_mbp10(TRAIN_DATES)
trades_train = load_trades(TRAIN_DATES)
print(f'Book rows: {len(book_train):,}  |  Trade rows: {len(trades_train):,}')

## 2. Feature Engineering (12 features per 1-minute bar)

In [ ]:
DECAY = 0.7   # exponential decay for multi-level features (from HW2)
N_LEVELS = 10


def compute_book_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute instantaneous book features from each MBP-10 snapshot.
    These are then aggregated (mean/last) into 1-minute bars.
    """
    out = pd.DataFrame(index=df.index)

    bid_px = np.column_stack([df[f'bid_px_{i:02d}'].values for i in range(N_LEVELS)])
    ask_px = np.column_stack([df[f'ask_px_{i:02d}'].values for i in range(N_LEVELS)])
    bid_sz = np.column_stack([df[f'bid_sz_{i:02d}'].values for i in range(N_LEVELS)])
    ask_sz = np.column_stack([df[f'ask_sz_{i:02d}'].values for i in range(N_LEVELS)])

    weights = np.array([DECAY**i for i in range(N_LEVELS)])

    # F1: Top-of-book OFI (L1)
    denom1 = (bid_sz[:, 0] + ask_sz[:, 0]).clip(min=1)
    out['ofi_l1'] = (bid_sz[:, 0] - ask_sz[:, 0]) / denom1

    # F2: Decay-weighted OFI (L2, 10 levels, from HW2)
    w_bid = (bid_sz * weights).sum(axis=1)
    w_ask = (ask_sz * weights).sum(axis=1)
    w_tot = (w_bid + w_ask).clip(min=1e-9)
    out['ofi_decay'] = (w_bid - w_ask) / w_tot

    # F3: Equal-weighted OFI (all 10 levels)
    eq_bid = bid_sz.sum(axis=1)
    eq_ask = ask_sz.sum(axis=1)
    eq_tot = (eq_bid + eq_ask).clip(min=1)
    out['ofi_equal'] = (eq_bid - eq_ask) / eq_tot

    # F4: Stoikov microprice (from HW2)
    b0, a0 = bid_px[:, 0], ask_px[:, 0]
    bs0, as0 = bid_sz[:, 0], ask_sz[:, 0]
    mid = (b0 + a0) / 2
    spread = (a0 - b0).clip(min=1e-6)
    microprice = (b0 * as0 + a0 * bs0) / (bs0 + as0).clip(min=1)
    out['microprice_norm'] = (microprice - mid) / (spread / 2)  # in [-1, +1]

    # F5: Bid-ask spread in bps
    out['spread_bps'] = (spread / mid.clip(min=0.01)) * 1e4

    # F6: Top-of-book depth (log shares at best bid + ask)
    out['depth_log'] = np.log1p(bid_sz[:, 0] + ask_sz[:, 0])

    # F7: Book slope — bid side (price impact per level)
    #     slope = regression of cumulative depth on tick distance from mid
    cum_bid = bid_sz.cumsum(axis=1)
    tick_dist = np.arange(1, N_LEVELS + 1).reshape(1, -1)
    # simple ratio: total depth across 5 levels / depth at top
    out['book_slope_bid'] = cum_bid[:, 4] / bid_sz[:, 0].clip(min=1)
    cum_ask = ask_sz.cumsum(axis=1)
    out['book_slope_ask'] = cum_ask[:, 4] / ask_sz[:, 0].clip(min=1)

    # F8: Queue imbalance (bid depth / ask depth — levels 0-4)
    bid5 = bid_sz[:, :5].sum(axis=1)
    ask5 = ask_sz[:, :5].sum(axis=1)
    out['queue_imb_5'] = (bid5 - ask5) / (bid5 + ask5).clip(min=1)

    # F9: Depth ratio (top-1 vs top-5)
    out['depth_ratio'] = bid_sz[:, 0] / bid5.clip(min=1)

    return out


print('Computing book features...')
tick_features = compute_book_features(book_train)
print(f'Tick features shape: {tick_features.shape}')

In [ ]:
def compute_trade_features(trades: pd.DataFrame) -> pd.Series:
    """
    From trades data: signed order flow imbalance per 1-minute bar.
    Databento trades have 'side' column: 'A' = ask-side (buyer-initiated), 'B' = bid-side.
    """
    if trades.empty:
        return pd.Series(dtype=float, name='trade_ofi')

    # signed size: +1 for buyer-initiated, -1 for seller-initiated
    signed = trades['size'] * trades['side'].map({'A': 1, 'B': -1, 'N': 0}).fillna(0)
    total  = trades['size']

    signed_1m = signed.resample('1min').sum()
    total_1m  = total.resample('1min').sum().replace(0, np.nan)
    return (signed_1m / total_1m).rename('trade_ofi')


# Aggregate book features to 1-minute bars
bar_features = tick_features.resample('1min').mean().dropna(how='all')

# Add trade OFI
trade_ofi = compute_trade_features(trades_train)
bar_features = bar_features.join(trade_ofi, how='left')
bar_features['trade_ofi'] = bar_features['trade_ofi'].fillna(0)

# Realized vol of midprice (from tick data, in bps)
mid_tick = (book_train['bid_px_00'] + book_train['ask_px_00']) / 2
mid_ret  = mid_tick.pct_change().dropna() * 1e4
rvol     = mid_ret.resample('1min').std().rename('rvol_bps')
bar_features = bar_features.join(rvol, how='left')

# Forward 1-minute return (the target)
mid_1m = mid_tick.resample('1min').last()
fwd_ret = mid_1m.pct_change(1).shift(-1) * 1e4   # in bps, no look-ahead
bar_features = bar_features.join(fwd_ret.rename('fwd_ret_bps'), how='left')

# Drop bars at market open (first 30 min) and rows with missing targets
bar_features = bar_features[
    (bar_features.index.time >= pd.Timestamp('10:00').time()) &
    (bar_features.index.time <  pd.Timestamp('15:45').time())
].dropna(subset=['fwd_ret_bps'])

print(f'1-minute bars: {len(bar_features):,}')
print(f'Features: {[c for c in bar_features.columns if c != "fwd_ret_bps"]}')

## 3. Information Coefficient (IC) per Feature

In [ ]:
FEATURE_COLS = [c for c in bar_features.columns if c != 'fwd_ret_bps']
TARGET = 'fwd_ret_bps'

ic = {}
for f in FEATURE_COLS:
    x = bar_features[f].dropna()
    y = bar_features[TARGET].reindex(x.index).dropna()
    x = x.reindex(y.index)
    ic[f], _ = stats.pearsonr(x, y)

ic_series = pd.Series(ic).sort_values(key=abs, ascending=False)
print('Information Coefficients (Pearson r with 1-min forward return):')
print(ic_series.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if v > 0 else 'tomato' for v in ic_series.values]
ax.barh(ic_series.index, ic_series.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('IC (Pearson r)')
ax.set_title('Feature IC vs 1-min forward return — Training set')
plt.tight_layout()
plt.savefig('figures/factor_zoo_ic.png', dpi=150)
plt.show()

## 4. OLS Model

In [ ]:
X_train = bar_features[FEATURE_COLS].dropna()
y_train = bar_features[TARGET].reindex(X_train.index)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

ols = LinearRegression()
ols.fit(X_scaled, y_train)

y_pred_train = ols.predict(X_scaled)
r2_train = r2_score(y_train, y_pred_train)
ic_train = np.corrcoef(y_pred_train, y_train)[0, 1]

print(f'OLS R² (train): {r2_train:.4f}')
print(f'OLS IC (train): {ic_train:.4f}')
print('\nCoefficients (scaled):')
coef_df = pd.DataFrame({'feature': FEATURE_COLS, 'coef': ols.coef_}).sort_values('coef', key=abs, ascending=False)
print(coef_df.to_string(index=False))

## 5. Gradient Boosting Model

In [ ]:
gbm = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)
gbm.fit(X_scaled, y_train)

y_pred_gbm_train = gbm.predict(X_scaled)
print(f'GBM R² (train): {r2_score(y_train, y_pred_gbm_train):.4f}')
print(f'GBM IC (train): {np.corrcoef(y_pred_gbm_train, y_train)[0,1]:.4f}')

# Feature importance
imp = pd.Series(gbm.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
imp.plot.barh(ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('GBM Feature Importance')
ax.set_title('Gradient Boosting Feature Importance — Training set')
plt.tight_layout()
plt.savefig('figures/factor_zoo_gbm_importance.png', dpi=150)
plt.show()

## 6. Ablation Study

In [ ]:
"""Leave-one-out ablation: train OLS on all-but-one feature, compare IC."""

ablation = {}
for drop_feat in FEATURE_COLS:
    remaining = [f for f in FEATURE_COLS if f != drop_feat]
    X_ab = scaler.fit_transform(X_train[remaining])
    m = LinearRegression().fit(X_ab, y_train)
    pred = m.predict(X_ab)
    ablation[f'drop {drop_feat}'] = np.corrcoef(pred, y_train)[0, 1]

full_ic = ic_train
ablation_df = pd.Series(ablation).sort_values()
ablation_df['full model'] = full_ic

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v < full_ic else 'steelblue' for v in ablation_df.values]
ablation_df.plot.barh(ax=ax, color=colors, alpha=0.8)
ax.axvline(full_ic, color='black', lw=1.5, ls='--', label='Full model IC')
ax.set_xlabel('OLS IC on training set')
ax.set_title('Ablation: IC when each feature is removed')
ax.legend()
plt.tight_layout()
plt.savefig('figures/factor_zoo_ablation.png', dpi=150)
plt.show()

print('Features that hurt IC most when removed (highest signal):',
      ablation_df.idxmin())

## 7. Out-of-Sample Test

In [ ]:
print('Loading test data...')
book_test   = load_mbp10(TEST_DATES)
trades_test_raw = load_trades(TEST_DATES)

tick_test  = compute_book_features(book_test)
bar_test   = tick_test.resample('1min').mean().dropna(how='all')
bar_test   = bar_test.join(compute_trade_features(trades_test_raw), how='left')
bar_test['trade_ofi'] = bar_test['trade_ofi'].fillna(0)

mid_test = (book_test['bid_px_00'] + book_test['ask_px_00']) / 2
rvol_test = mid_test.pct_change().dropna().resample('1min').std().rename('rvol_bps') * 1e4
bar_test = bar_test.join(rvol_test, how='left')
fwd_test = mid_test.resample('1min').last().pct_change(1).shift(-1) * 1e4
bar_test = bar_test.join(fwd_test.rename('fwd_ret_bps'), how='left')
bar_test = bar_test[
    (bar_test.index.time >= pd.Timestamp('10:00').time()) &
    (bar_test.index.time <  pd.Timestamp('15:45').time())
].dropna(subset=['fwd_ret_bps'])

X_test = bar_test[FEATURE_COLS].dropna()
y_test = bar_test['fwd_ret_bps'].reindex(X_test.index)
X_test_scaled = scaler.transform(X_test)  # use TRAINING scaler — no look-ahead

pred_ols_test = ols.predict(X_test_scaled)
pred_gbm_test = gbm.predict(X_test_scaled)

print(f'OLS  R² (test): {r2_score(y_test, pred_ols_test):.4f}   IC: {np.corrcoef(pred_ols_test, y_test)[0,1]:.4f}')
print(f'GBM  R² (test): {r2_score(y_test, pred_gbm_test):.4f}   IC: {np.corrcoef(pred_gbm_test, y_test)[0,1]:.4f}')

## 8. Signal → Strategy (top-quartile OLS predictions)

In [ ]:
"""
Trading rule: enter long when predicted return is in top quartile,
short when in bottom quartile. Hold 1 bar (1 min). Exit next open.
TC = half spread per side.
"""
SHARES = 100

def signal_backtest(bar_df: pd.DataFrame, predictions: np.ndarray, shares: int = SHARES) -> pd.DataFrame:
    df = bar_df.copy()
    df['pred'] = predictions

    q75 = np.percentile(predictions, 75)
    q25 = np.percentile(predictions, 25)

    mid_series = (book_train['bid_px_00'] + book_train['ask_px_00']) / 2
    spread_series = (book_train['ask_px_00'] - book_train['bid_px_00'])

    mid_1m    = mid_series.resample('1min').last().reindex(df.index)
    spread_1m = spread_series.resample('1min').mean().reindex(df.index)

    trades = []
    for i in range(len(df) - 1):
        pred  = df['pred'].iloc[i]
        if pred >= q75:
            direction = 1
        elif pred <= q25:
            direction = -1
        else:
            continue

        t_entry = df.index[i + 1]
        t_exit  = df.index[i + 2] if i + 2 < len(df) else None
        if t_exit is None:
            continue

        ep = mid_1m.get(t_entry, np.nan)
        xp = mid_1m.get(t_exit,  np.nan)
        sp_entry = spread_1m.get(t_entry, np.nan)
        sp_exit  = spread_1m.get(t_exit,  np.nan)

        if np.isnan(ep) or np.isnan(xp):
            continue

        tc = (sp_entry + sp_exit) / 2 * shares
        gross = direction * (xp - ep) * shares
        trades.append({
            'entry_time':  t_entry,
            'exit_time':   t_exit,
            'direction':   direction,
            'gross_pnl':   gross,
            'tc':          tc,
            'net_pnl':     gross - tc,
        })

    return pd.DataFrame(trades)


strat_train = signal_backtest(bar_test, pred_ols_test)
print(f'Trades: {len(strat_train)}   Total PnL: ${strat_train.net_pnl.sum():.2f}'
      f'   Hit rate: {(strat_train.net_pnl > 0).mean():.1%}')

## 9. Summary: IC Table (Train vs Test)

In [ ]:
ic_test = {}
for i, f in enumerate(FEATURE_COLS):
    x = X_test[f]
    r, _ = stats.pearsonr(x, y_test.reindex(x.index).dropna())
    ic_test[f] = r

summary = pd.DataFrame({
    'IC_train': pd.Series(ic),
    'IC_test':  pd.Series(ic_test),
}).sort_values('IC_train', key=abs, ascending=False)

print(summary.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(summary))
ax.bar(x_pos - 0.2, summary['IC_train'], width=0.4, label='Train', color='steelblue', alpha=0.8)
ax.bar(x_pos + 0.2, summary['IC_test'],  width=0.4, label='Test',  color='darkorange', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(summary.index, rotation=45, ha='right')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('IC (Pearson r)')
ax.set_title('Feature IC — Train vs Test')
ax.legend()
plt.tight_layout()
plt.savefig('figures/factor_zoo_ic_comparison.png', dpi=150)
plt.show()